# specdec on a T4: Qwen2.5-0.5B drafting for Qwen2.5-3B, plus n-gram speculation

Baseline, draft-model speculation at k = 1, 3, 5, and n-gram speculation, each at concurrency 1, 4 and 16; acceptance rate from vLLM's counters; a byte-identity check of greedy outputs. ~10 min install + ~6 min per configuration (5 configurations).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# Install recipe from inference-infra-portfolio/KAGGLE.md
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/specdec" "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, specdec.analysis, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## Configurations

In [ ]:
import asyncio, json, subprocess, time, httpx
from servellm.launch import ServerConfig, VllmServer
from specdec.analysis import parse_counters, acceptance_from_counters, spec_config_arg

TARGET, DRAFT = "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"
CONFIGS = [("baseline", 0, ())]
CONFIGS += [(f"draft k={k}", k, tuple(spec_config_arg("draft_model", k, model=DRAFT))) for k in (1, 3, 5)]
CONFIGS += [("ngram k=4", 4, tuple(spec_config_arg("ngram", 4, prompt_lookup_min=2, prompt_lookup_max=5)))]
LEVELS = [1, 4, 16]
BASE = "http://127.0.0.1:8000"

def metrics():
    return parse_counters(httpx.get(f"{BASE}/metrics", timeout=10).text)

GREEDY_PROMPTS = ["Explain the difference between TCP and UDP in five sentences.",
                  "Write a Python function that returns the n-th Fibonacci number iteratively, with a docstring.",
                  "Summarise the causes of the French Revolution in one paragraph."]

def greedy_outputs(model):
    outs = []
    for p in GREEDY_PROMPTS:
        r = httpx.post(f"{BASE}/v1/chat/completions", json={"model": model, "messages": [{"role": "user", "content": p}],
                       "max_tokens": 160, "temperature": 0, "seed": 0}, timeout=None).json()
        outs.append(r["choices"][0]["message"]["content"])
    return outs

In [ ]:
runs, identity = [], {}
for name, k, extra in CONFIGS:
    tag = name.replace(" ", "_").replace("=", "")
    server = VllmServer(ServerConfig(model=TARGET, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, extra_args=extra),
                        log_path=f"results/startup_{tag}.log")
    try:
        info = server.start(ready_timeout_s=1200)
        print(name, "| KV cache tokens:", info.kv_cache_tokens, "| load GiB:", info.model_load_gib)
        identity[name] = greedy_outputs(TARGET)
        for conc in LEVELS:
            before = metrics()
            subprocess.run(["tokenbench", "sweep", "--base-url", BASE, "--model", TARGET, "--out", f"results/sweep_{tag}_c{conc}.json",
                            "--levels", str(conc), "--num-requests", str(max(32, 4 * conc)), "--warmup", "2",
                            "--prompts", "sample", "--max-tokens", "192", "--no-ignore-eos"], check=True)
            after = metrics()
            lv = json.load(open(f"results/sweep_{tag}_c{conc}.json"))["levels"][0]
            runs.append({"config": name, "k": k, "concurrency": conc, "itl_p50_ms": lv["itl"]["p50"] * 1000, "itl_p95_ms": lv["itl"]["p95"] * 1000,
                         "ttft_p50_ms": lv["ttft"]["p50"] * 1000, "output_tokens_per_s": lv["output_tokens_per_s"], "requests": lv["requests"],
                         "kv_cache_tokens": info.kv_cache_tokens, "acceptance": acceptance_from_counters(before, after)})
            print(f"  conc={conc}: ITL p50 {runs[-1]['itl_p50_ms']:.1f} ms, {runs[-1]['output_tokens_per_s']:.0f} tok/s, acceptance {runs[-1]['acceptance']['acceptance_rate']}")
    except Exception as e:
        print(name, "FAILED:", type(e).__name__, e); runs.append({"config": name, "k": k, "concurrency": None, "error": f"{type(e).__name__}: {e}"})
    finally:
        server.stop(); time.sleep(5)
json.dump(runs, open("results/runs.json", "w"), indent=1); json.dump(identity, open("results/greedy_outputs.json", "w"), indent=1)

## Results

In [ ]:
from specdec.analysis import build_table, render_markdown
from IPython.display import Markdown, display
ok = [r for r in runs if "error" not in r]
rows = build_table(ok); md = render_markdown(rows)
open("results/comparison.md", "w").write(md); json.dump(rows, open("results/comparison.json", "w"), indent=1)
display(Markdown(md))
base = identity.get("baseline")
for name, outs in identity.items():
    same = base is not None and outs == base
    print(f"greedy outputs identical to baseline: {name:12s} {same}")
json.dump({n: (o == base) for n, o in identity.items()}, open("results/identity_check.json", "w"), indent=1)

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name in dict.fromkeys(r["config"] for r in ok):
    rs = sorted((r for r in ok if r["config"] == name), key=lambda r: r["concurrency"])
    axes[0].plot([r["concurrency"] for r in rs], [r["itl_p50_ms"] for r in rs], marker="o", label=name)
    axes[1].plot([r["concurrency"] for r in rs], [r["output_tokens_per_s"] for r in rs], marker="o", label=name)
for ax, yl, t in ((axes[0], "ITL p50 (ms)", "Decode latency"), (axes[1], "output tokens / s", "Throughput")):
    ax.set_xscale("log", base=2); ax.set_xticks(LEVELS); ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("concurrent requests"); ax.set_ylabel(yl); ax.set_title(t); ax.grid(alpha=.3); ax.legend()
fig.suptitle("Speculative decoding on a T4: Qwen2.5-0.5B draft, Qwen2.5-3B target"); fig.tight_layout(); fig.savefig("results/spec_vs_concurrency.png", dpi=130)
from IPython.display import Image; display(Image("results/spec_vs_concurrency.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip